# __AutoGrad__

- autograd는 연산을 할 때마다 `미분 그래프(computation graph)`를 자동으로 만들어서 역전파로 gradient 계산해주는 시스템이다.

## __1.Tensor + requires_grad__

- 아래 코드는 해당 tensor에 대해 grad 추적 기능을 활성화 한다. 이후 연산이 모두 기록되므로, 메모리 사용에 유의해야 한다. `requires_grad는 float type만 지원한다 소수 범위의 변화율을 확인하기 위해서이다.`

In [7]:
import torch

x = torch.tensor([2.0], requires_grad = True)
print(x)

tensor([2.], requires_grad=True)


- 실제 requires_grad를 통해 역전파를 연산해보자.

In [8]:
y = x * 3
z = y ** 2

z.backward()
print(x.grad)

tensor([36.])


## __2.Structure autograd__

- Tensor는 autograd 관하여 3개의 정보를 갖는다.
    - data : 실제 값
    - grad : 미분값 저장 공간
    - grad_fn : 어떤 연산으로 만들어졌는지

In [ ]:
print(z.grad_fn) #pow는 power로 거듭제곱의 연산으로 만들어 졌음을 의미.

## __3.Leaf Tensor vs Non-leaf Tensor__

- leaf tensor: 사용자가 생성한 tensor로, grad가 저장된다.
- non-leaf tensor: 연산 결과로, 기본적으로 grad 저장을 하지 않는다.

`두 종류의 tensor를 기반으로 이해하자.`
- (1) 그래프 끝 z 부터 시작한다.
- (2) chain rule로 뒤로(x)로 이동하면서
- (3) 각 노드의 gradient를 계산한다.
- (4) leaf tensor의 .grad에 저장한다.

In [10]:
x = torch.tensor(2.0, requires_grad = True) #leaf
y = x * 3 #non-leaf
print(x.grad)
print(y.grad) #error

None
None


/var/folders/vf/d0gygbv5527dd2h9xypg_1zm0000gn/T/ipykernel_1354/620341740.py:4: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  print(y.grad) #error


## __4.gradient의 누적__

- gradient는 기본적으로 누적된다. 아래 예시를 통해 이해하자.

In [ ]:
x = torch.tensor(2.0, requires_grad = True)
y = x * 3
y.backward() #3
print(x.grad)

y = x * 4
y.backward() #4
print(x.grad) #3 + 4

tensor(3.)
tensor(7.)
